In [1]:
print("Streaming")

Streaming


In [2]:
from dotenv import load_dotenv
load_dotenv()
import os 
from langchain.chat_models import init_chat_model

In [3]:
model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
)

for chunk in model.stream("typescript vs rust which is dominating>"):
    print(chunk.content, end="", flush=True)

## TL;DR  

| Metric (2023‑2024) | **TypeScript** | **Rust** |
|-------------------|----------------|----------|
| **Overall popularity** (Stack Overflow 2023, GitHub 2024) | #2 most‑used language (≈ 30 % of respondents) | #13 most‑used (≈ 5 % of respondents) |
| **Growth rate** (GitHub “language‑specific commits”) | +12 % YoY | +28 % YoY |
| **Job postings (US, major boards)** | ~45 k openings (≈ 30 % of all dev jobs) | ~7 k openings (≈ 5 % of all dev jobs) |
| **Top domains** | Front‑end web, Node.js back‑ends, serverless, desktop (Electron), cross‑platform mobile (React‑Native) | Systems programming, embedded, WebAssembly, networking, security‑critical services, CLI tools |
| **Community size** (npm downloads vs crates.io downloads) | ~2 billion npm downloads / month | ~140 million crates.io downloads / month |
| **Salary premium (US)** | +5 % vs median dev salary | +15 % vs median dev salary |
| **Learning curve** | Low‑to‑medium (JS background) | Medium‑to‑high (systems concepts, 

In [4]:
from langchain_core.messages import AIMessageChunk

In [5]:
# Basic streaming — simplest form
for chunk in model.stream("Explain gradient descent step by step."):
    print(chunk.content, end="", flush=True)

print()   # newline after streaming finishes

**Gradient Descent – A Step‑by‑Step Walkthrough**

Gradient descent is one of the most widely used optimization algorithms in machine learning, statistics, and many areas of engineering.  
Its goal is simple: *find the parameters (weights) that minimize a given loss (cost) function*.

Below is a complete, intuitive, and mathematically grounded guide that walks you through the whole process—from the problem statement to a working implementation.

---

## 1. The Setting

### 1.1. What are we trying to minimize?
Suppose we have a **loss function** (also called objective or cost function)

\[
J(\theta) \;:\; \mathbb{R}^d \to \mathbb{R}
\]

where  

* \(\theta = (\theta_1,\dots,\theta_d)\) are the parameters we can adjust (e.g., weights of a linear model).  
* \(J(\theta)\) measures how badly the model fits the data (e.g., mean‑squared error, cross‑entropy).

Our task: **find the \(\theta^\* \) that makes \(J(\theta)\) as small as possible**.

### 1.2. Why not just solve it analytically?
Fo

In [6]:
    # if full is None:
    #     full = chunk
    # # On all subsequent chunks, add the new chunk to the accumulated full
    # else:
    #     full = full + chunk

# full = chunk if full is None else full + chunk

In [7]:
full = None   # will be AIMessageChunk or None

for chunk in model.stream("What is machine learning?"):
    # Add chunks together to build the full message
    full = chunk if full is None else full + chunk
    print(chunk.content, end="", flush=True)

print()

# After the loop, full is a complete assembled AIMessageChunk
# It behaves exactly like an AIMessage
print("\n--- ASSEMBLED MESSAGE ---")
print("type           :", type(full))
print("content        :", full.content[:80])  #type:ignore
print("usage_metadata :", full.usage_metadata) #type:ignore
print("response_meta  :", full.response_metadata) #type:ignore

**Machine learning (ML)** is a subfield of artificial intelligence (AI) that focuses on giving computers the ability to learn from data and improve their performance on a task without being explicitly programmed for every possible situation.

### Core Idea
- **Traditional programming:** You write explicit rules (if‑then statements) that tell the computer exactly what to do.
- **Machine learning:** You provide the computer with data and a goal (e.g., predict something, classify items, generate text). The algorithm automatically discovers patterns or rules from the data that help it achieve the goal.

### How It Works (Simplified)
1. **Collect data** – a set of examples (e.g., images of cats, transaction records, sensor readings).
2. **Choose a model** – a mathematical structure (like a decision tree, neural network, or linear regression) that can represent relationships in the data.
3. **Training** – feed the data to the model and let an optimization process adjust the model’s internal 

In [8]:
# Streaming with Conversation History

In [9]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")

conversation = [
    SystemMessage("You are a Python tutor. Be concise."),
    HumanMessage("What is a decorator?"),
    AIMessage("A decorator wraps a function to modify its behavior without changing its code."),
    HumanMessage("Can you show me a simple example?")
]

for chunk in model.stream(conversation):
    print(chunk.content, end="", flush=True)

print()

Sure! Here’s a minimal decorator that prints a message **before** and **after** a function runs.

```python
def my_decorator(func):
    """Wraps *func* with extra behavior."""
    def wrapper(*args, **kwargs):
        print(">>> Before the call")
        result = func(*args, **kwargs)   # run the original function
        print(">>> After the call")
        return result
    return wrapper


@my_decorator          # ← applies the decorator
def greet(name):
    print(f"Hello, {name}!")


# Use it
greet("Alice")
```

**Output**

```
>>> Before the call
Hello, Alice!
>>> After the call
```

- `my_decorator` receives the original function (`greet`) and returns a new function (`wrapper`) that adds extra steps.
- The `@my_decorator` syntax is just syntactic sugar for `greet = my_decorator(greet)`.


In [10]:
# Content Blocks in Stream — The Right Way to Read Chunks

In [14]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")

for chunk in model.stream("What color is the sky?"):
    for block in chunk.content_blocks:

        if block['type'] == "text":
            print(block['text'], end="", flush=True)

        elif block["type"] == "reasoning":
            # Thinking tokens (Claude extended thinking / o-series reasoning)
            reasoning_text = block.get("reasoning", "")
            if reasoning_text:
                print(f"{reasoning_text}", end="", flush=True)

        elif block["type"] == "tool_call_chunk":
            # Partial tool call arriving piece by piece
            print(f"\nTOOL CHUNK name={block.get('name')} args={block.get('args')}")

print()

The user asks a simple question: "What color is the sky?" The answer: typically blue, but can vary. Provide answer. No policy issues.The sky is most commonly described as **blue** during clear daylight hours, due to the way Earth's atmosphere scatters sunlight. However, its color can change depending on the time of day and atmospheric conditions—ranging from reds and oranges at sunrise or sunset, to deep black at night, and even shades of gray or white when it’s overcast or foggy.


<style>
  /* Import Fira Code from Google Fonts */
  @import url('https://fonts.googleapis.com/css2?family=Fira+Code:wght@400;700&display=swap');
</style>

<div class="alert alert-success" style="font-family: 'Fira Code', monospace; padding: 15px; border-radius: 5px; background-color: #d4edda; color: #155724; border: 1px solid #c3e6cb;">
  <strong>Note:</strong> When streaming, tool calls arrive in parts.chunk.tool_call_chunks to capture them.
</div>

In [16]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import AIMessageChunk, BaseMessage, HumanMessage, ToolMessage

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

tools = [get_weather]
tools_by_name = {t.name: t for t in tools}

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
llm = model.bind_tools(tools)

messages: list[BaseMessage] = [HumanMessage("What is the weather in Paris?")]

# step 1 — model decides which tool to call
accumulated = None
for chunk in llm.stream(messages):
    if isinstance(chunk, AIMessageChunk):
        accumulated = chunk if accumulated is None else accumulated + chunk

if accumulated is None:
    raise RuntimeError("No AIMessageChunk received from model.")

print("tool_calls:", accumulated.tool_calls)
messages.append(accumulated)  # add AI message to history

# step 2 — execute each tool the model requested
for tool_call in accumulated.tool_calls:
    tool_fn  = tools_by_name[tool_call["name"]]
    result   = tool_fn.invoke(tool_call["args"])
    messages.append(ToolMessage(content=result, tool_call_id=tool_call["id"]))
    print(f"tool result: {result}")

# step 3 — send tool results back, get final text response
for chunk in llm.stream(messages):
    if isinstance(chunk, AIMessageChunk):
        print(chunk.content, end="", flush=True)

print()

tool_calls: [{'name': 'get_weather', 'args': {'city': 'Paris'}, 'id': 'fc_d3960cfd-21ad-4662-b3fb-4f766910bc1c', 'type': 'tool_call'}]
tool result: It's always sunny in Paris!
The current weather in Paris is sunny. Enjoy your day!


In [17]:
# Streaming Reasoning Tokens (Claude Extended Thinking)

In [19]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from dotenv import load_dotenv
load_dotenv()

# Nemotron 3 Super — efficient reasoning and agentic tasks
model = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")


reasoning_parts = []
text_parts = []

for chunk in model.stream("Solve: if x^2 = 16 and x > 0, what is x?"):
    for block in chunk.content_blocks:
        if block["type"] == "reasoning":
            r = block.get("reasoning", "")
            if r:
                reasoning_parts.append(r)
                print(f"{r}", end="", flush=True)
        elif block["type"] == "text":
            t = block.get("text", "")
            if t:
                text_parts.append(t)
                print(t, end="", flush=True)

print()
print("Reasoning chars:", sum(len(r) for r in reasoning_parts))
print("Answer         :", "".join(text_parts))

Okay, the user asks:

"Solve: if x^2 = 16 and x > 0, what is x?"

We need to solve x^2 = 16 => x = ±4, but given x > 0, we have x = 4.

Thus answer: x = 4.

We need to produce a final answer.

\(x = 4\)
Reasoning chars: 193
Answer         : \(x = 4\)


In [20]:
# astream — Async Streaming

In [24]:
import asyncio
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from dotenv import load_dotenv
load_dotenv()

# Nemotron 3 Super — efficient reasoning and agentic tasks
model = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")

async def stream_async():
    async for chunk in model.astream("Write a haiku about rain."):
        print(chunk.content, end="", flush=True)
    print()

await stream_async()

Okay, the user wants a haiku about rain. A haiku is a traditional Japanese poem with 5-7-5 syllable structure. I need to create a short poem that captures the essence of rain, following that syllable count.

First, I should brainstorm some rain-related imagery: raindrops falling, the sound of rain, how it affects the environment, feelings it evokes, etc. Then, I'll craft lines that fit 5, 7, 5 syllables.

Let me think of the first line (5 syllables). Examples: "Soft drops kiss the earth" – let's count: Soft(1) drops(2) kiss(3) the(4) earth(5). That's 5. Good.

Second line (7 syllables): Needs to be 7. Maybe something like "Whispering through leaves, a cool embrace" – count: Whis-per-ing(3) through(4) leaves(5), a(6) cool(7) em-brace(8). That's 8, too many. Let me adjust.

"Whispering through leaves, cool and slow" – Whis-per-ing(3) through(4) leaves(5), cool(6) and(7) slow(8). Still 8. "Whispering leaves, cool and slow" – Whis-per-ing(3) leaves(4), cool(5) and(6) slow(7). That's 7. But

In [25]:
async for chunk in model.astream("Write a haiku about rain."):
    print(chunk.content, end="", flush=True)
print()

Okay, the user asks for a haiku about rain. A haiku is a traditional Japanese poem with 5 syllables in the first line, 7 in the second, and 5 in the third. It often captures a moment in nature, so rain fits perfectly.

I need to create something simple, evocative, and true to the form. Rain can be gentle, soothing, or powerful—I'll go for a calm, reflective image to keep it peaceful.

Brainstorming lines: First line (5 syllables) could be about the rain starting. Like "Soft drops fall from sky" – let's count: Soft(1) drops(2) fall(3) from(4) sky(5). That's 5.

Second line (7 syllables): Needs to describe the effect. Maybe "On leaves, they whisper secrets" – On(1) leaves,(2) they(3) whis(4)per(5) se(6)crets(7). "Whispers" is two syllables: whis-pers, so "whisper" would be 2, but I said "whisper" as one? Wait, no: "whisper" is 2 syllables (whis-per), so "they whisper" is they(1) whis(2) per(3)? Better to break it down.

Standard way: "they whisper" – they (1), whis (2), per (3) – so 3 sy

In [26]:
# astream with Content Blocks (Async)

In [27]:
async def stream_with_blocks():
    async for chunk in model.astream("Explain what a neural network is."):
        for block in chunk.content_blocks:
            if block["type"] == "text":
                print(block["text"], end="", flush=True)
            elif block["type"] == "reasoning":
                reasoning = block.get("reasoning", "")
                if reasoning:
                    print(f"{reasoning}", end="")
    print()

await stream_with_blocks()

Okay,0.0
Okay, the user is asking me to explain what a neural network is. This seems like a foundational question, probably from someone new to machine learning or AI. I should keep it clear and accessible without oversimplifying.

Hmm, they might be a student, a curious hobbyist, or maybe a professional looking to refresh basics. Since they didn't specify technical depth, I'll aim for a balanced explanation - concrete enough to grasp the concept but avoiding heavy math jargon. 

I recall they might actually be wondering: "How do these things actually learn?" or "Why should I care about this buzzword?" So I should connect the explanation to real-world impact - like how neural networks power things they use daily (voice assistants, recommendations).

Important to emphasize it's inspired by brains but not identical - that's a common misconception. Should clarify it's about mathematical functions approximating patterns, not actual neurons firing. 

The layered structure is key to explain:

In [28]:
# astream_events — Semantic Event Stream

In [29]:
async def stream_events():
    async for event in model.astream_events("Tell me a fun fact.", version="v2"):
        kind = event["event"]
        data = event.get("data", {})

        if kind == "on_chat_model_start":
            print("[START] Model started generating")
            model_input = data.get("input")
            if model_input is not None:
                print("        Input:", model_input)

        elif kind == "on_chat_model_stream":
            chunk = data.get("chunk")
            if chunk is not None:
                token = chunk.content
                if token:
                    print(token, end="", flush=True)

        elif kind == "on_chat_model_end":
            print("\n[END] Model finished")
            output = data.get("output")
            if output is not None:
                print("      usage_metadata :", output.usage_metadata)
                print("      finish_reason  :", output.response_metadata.get("finish_reason"))

await stream_events()

[START] Model started generating
        Input: Tell me a fun fact.
Here’s a fun one: A single bolt of lightning contains enough energy to toast about 100,000 slices of bread! 🌩️🍞
[END] Model finished
      usage_metadata : {'input_tokens': 22, 'output_tokens': 75, 'total_tokens': 97}
      finish_reason  : stop


In [30]:
import json

chunk_count = 0

for chunk in model.stream("What is deep learning?"):
    chunk_count += 1
    print(f"\n{'———'*40}")
    print(f"CHUNK #{chunk_count}")
    print(f"\n{'———'*40}")

    print(f"type                : {type(chunk).__name__}")
    print(f".type               : {chunk.type}")
    print(f".id                 : {chunk.id}")
    print(f".content            : {repr(chunk.content)}")

    # Tool call chunks
    print(f".tool_call_chunks   : {chunk.tool_call_chunks}")
    print(f".tool_calls         : {chunk.tool_calls}")

    # Content blocks
    print(f".content_blocks     : {chunk.content_blocks}")

    # Metadata (usually empty until last chunk)
    print(f".usage_metadata     : {chunk.usage_metadata}")
    print(f".response_metadata  : {chunk.response_metadata}")
    print(f".additional_kwargs  : {chunk.additional_kwargs}")

print(f"\nTotal chunks received: {chunk_count}")


————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
CHUNK #1

————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
type                : AIMessageChunk
.type               : AIMessageChunk
.id                 : lc_run--019d8156-c518-7791-b3b0-83850c4d1f6f
.content            : ''
.tool_call_chunks   : []
.tool_calls         : []
.content_blocks     : []
.usage_metadata     : None
.response_metadata  : {}
.additional_kwargs  : {}

————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
CHUNK #2

————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
type                : AIMessageChunk
.type               : AIMessageChunk
.id                 : lc_run--019d8156-c518-7791-b3b0-83850c4d1f6f
.content            : ''
.tool_call

In [31]:
# Watching usage_metadata Appear on the Last Chunk

last_chunk = None

for i, chunk in enumerate(model.stream("What is the capital of Japan?")):
    last_chunk = chunk
    has_usage = chunk.usage_metadata is not None
    has_finish = bool(chunk.response_metadata.get("finish_reason"))
    print(f"Chunk {i:03d} | content={repr(chunk.content):<20} | has_usage={has_usage} | has_finish={has_finish}")

if last_chunk is None:
    raise RuntimeError("No chunks were returned by the model.")

print("\n--- LAST CHUNK METADATA ---")
print("usage_metadata    :", last_chunk.usage_metadata) 
print("response_metadata :", last_chunk.response_metadata)

Chunk 000 | content=''                   | has_usage=False | has_finish=False
Chunk 001 | content=''                   | has_usage=False | has_finish=False
Chunk 002 | content='The capital of Japan is Tokyo.' | has_usage=False | has_finish=False
Chunk 003 | content=''                   | has_usage=False | has_finish=True
Chunk 004 | content=''                   | has_usage=True | has_finish=False
Chunk 005 | content=''                   | has_usage=False | has_finish=False

--- LAST CHUNK METADATA ---
usage_metadata    : None
response_metadata : {}


In [32]:
full: AIMessageChunk | None = None

for chunk in model.stream("What is Python used for?"):
    full = chunk if full is None else full + chunk
    print(chunk.content, end="", flush=True)

if full is None:
    raise RuntimeError("No chunks were returned by the model.")

print("\n")


print("type              :", type(full).__name__)
print("content           :", full.content[:100])
print("usage_metadata    :")
print(json.dumps(full.usage_metadata, indent=2, default=str))
print("response_metadata :")
print(json.dumps(full.response_metadata, indent=2, default=str))
print("tool_calls        :", full.tool_calls)
print("content_blocks    :", full.content_blocks)

Okay, Python is a versatile, high-level programming language used in a wide range of applications due to its simplicity, readability, and extensive libraries. Here are some of the most common uses:

1. **Web Development**  
   - Frameworks like **Django** and **Flask** enable rapid development of secure, scalable web applications.
   - Used by companies like Instagram, Pinterest, and Spotify.

2. **Data Science & Analytics**  
   - Libraries such as **NumPy**, **pandas**, **Matplotlib**, and **SciPy** make data manipulation, analysis, and visualization efficient.
   - Essential for data cleaning, exploratory data analysis (EDA), and statistical modeling.

3. **Machine Learning & Artificial Intelligence**  
   - Dominant in ML/AI with libraries like **TensorFlow**, **PyTorch**, **scikit-learn**, and **Keras**.
   - Used for building neural networks, natural language processing (NLP), computer vision, and predictive models.

4. **Automation & Scripting**  
   - Ideal for writing scripts 

In [ ]:
# Streaming with System Prompt

from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("You are an expert Python teacher. Be concise and use code examples."),
    HumanMessage("Explain list comprehensions.")
]

for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

print()

In [33]:
# Streaming with Multiple Providers

In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangChain | Multi-Provider Streaming
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain.chat_models import init_chat_model
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import AIMessageChunk
from dotenv import load_dotenv

load_dotenv()

# ── question ─────────────────────────────────────────────────────────
question = "What is the capital of France? One sentence."

# ── model 1: NVIDIA Nemotron (direct class, no init_chat_model) ──────
def stream_nvidia():
    model = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")  # reasoning + agentic
    for chunk in model.stream(question):
        print(chunk.content, end="", flush=True)                   # stream token by token
    print()

# ── model 2: Groq via init_chat_model (returns AIMessageChunk) ───────
def stream_groq():
    model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
    for chunk in model.stream(question):
        if isinstance(chunk, AIMessageChunk):                      # type guard required
            print(chunk.content, end="", flush=True)
    print()

# ── model 3: OpenRouter GLM (direct class, no init_chat_model) ───────
def stream_openrouter():
    model = ChatOpenRouter(
        model="z-ai/glm-5.1",
        temperature=0.7,
        max_tokens=500,
    )
    for chunk in model.stream(question):
        print(chunk.content, end="", flush=True)                   # stream token by token
    print()

# ── run all providers ─────────────────────────────────────────────────
stream_nvidia()      # NVIDIA_API_KEY
stream_groq()        # GROQ_API_KEY
stream_openrouter()  # OPENROUTER_API_KEY


--- nvidia/nemotron-3-super-120b-a12b ---
The capital of France is Paris.

--- groq/gpt-oss-120b ---
Paris is the capital of France.

--- z-ai/glm-5.1 ---
The capital of France is Paris.


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangChain | Parallel Async Streaming with Reasoning
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import asyncio
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_openrouter import ChatOpenRouter
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

# ── question ──────────────────────────────────────────────────────────
question = "What is the capital of France? One sentence."

# ── helper: extract token from chunk ─────────────────────────────────
def extract_tokens(chunk):
    reasoning_token = ""
    text_token      = ""
    for block in getattr(chunk, "content_blocks", []):
        if block.get("type") == "reasoning":
            reasoning_token += block.get("reasoning", "") or block.get("text", "")
        elif block.get("type") == "text":
            text_token += block.get("text", "")
    if not reasoning_token and not text_token:    # fallback for plain string
        text_token = chunk.content or ""
    return reasoning_token, text_token

# ── generic async streamer ────────────────────────────────────────────
async def stream(label, model):
    reasoning_buf = ""
    response_buf  = ""
    async for chunk in model.astream(question):
        r, t        = extract_tokens(chunk)
        reasoning_buf += r
        response_buf  += t
    # print after done to avoid interleaved output from parallel runs
    print(f"\n{'━'*60}")
    print(f"  {label}")
    print(f"{'━'*60}")
    if reasoning_buf:
        print(f"[thinking]\n{reasoning_buf.strip()}\n")
    print(f"[answer]\n{response_buf.strip()}")

# ── models ────────────────────────────────────────────────────────────
nvidia_model      = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b").with_thinking_mode(enabled=True)
groq_model        = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
openrouter_model  = ChatOpenRouter(model="qwen/qwen3.6-plus-preview:free", max_tokens=8000)

# ── run all three in parallel ─────────────────────────────────────────
async def main():
    await asyncio.gather(
        stream("nvidia/nemotron-3-super-120b-a12b",    nvidia_model),
        stream("groq/gpt-oss-120b",                    groq_model),
        stream("qwen/qwen3.6-plus-preview",            openrouter_model),
    )

# await main()

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangChain | Parallel Async Streaming — Jupyter Notebook Run
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import asyncio
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_openrouter import ChatOpenRouter
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

# ── question ──────────────────────────────────────────────────────────
question = "What is the capital of France? One sentence."

# ── helper: extract reasoning and text tokens from chunk ──────────────
def extract_tokens(chunk):
    reasoning_token, text_token = "", ""
    for block in getattr(chunk, "content_blocks", []):
        if block.get("type") == "reasoning":
            reasoning_token += block.get("reasoning", "") or block.get("text", "")
        elif block.get("type") == "text":
            text_token += block.get("text", "")
    if not reasoning_token and not text_token:
        text_token = chunk.content or ""
    return reasoning_token, text_token

# ── generic async streamer ────────────────────────────────────────────
async def stream(label, model):
    reasoning_buf, response_buf = "", ""
    async for chunk in model.astream(question):
        r, t           = extract_tokens(chunk)
        reasoning_buf += r
        response_buf  += t
    print(f"\n{'━'*60}")
    print(f"  {label}")
    print(f"{'━'*60}")
    if reasoning_buf:
        print(f"[thinking]\n{reasoning_buf.strip()}\n")
    print(f"[answer]\n{response_buf.strip()}")

# ── models ────────────────────────────────────────────────────────────
nvidia_model     = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b").with_thinking_mode(enabled=True)
groq_model       = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
# openrouter_model = ChatOpenRouter(model="qwen/qwen3.6-plus-preview:free", max_tokens=8000)
# openrouter_model = ChatOpenRouter(model="google/gemma-4-31b-it:free", max_tokens=8000)
# openrouter_model = ChatOpenRouter(model="minimax/minimax-m2.5:free", max_tokens=8000)

# ── run in parallel — works directly in Jupyter ───────────────────────
await asyncio.gather(
    stream("nvidia/nemotron-3-super-120b-a12b", nvidia_model),
    stream("groq/gpt-oss-120b",                 groq_model),
    # stream("minimax/minimax-m2.5:free",         openrouter_model),
)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  groq/gpt-oss-120b
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[thinking]
The user asks: "What is the capital of France? One sentence." So answer: "Paris is the capital of France." One sentence.

[answer]
Paris is the capital of France.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  nvidia/nemotron-3-super-120b-a12b
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[thinking]
User asks: "What is the capital of France? One sentence." So answer: "The capital of France is Paris." One sentence.

[answer]
The capital of France is Paris.


In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangChain | Batch Processing — Jupyter Notebook Run
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain.chat_models import init_chat_model
from langchain_core.language_models import LanguageModelInput
from typing import cast
from dotenv import load_dotenv

load_dotenv()

# ── questions ─────────────────────────────────────────────────────────
questions = [
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?",
]

# ── models ────────────────────────────────────────────────────────────
nvidia_model = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b").with_thinking_mode(enabled=True)
groq_model   = init_chat_model("openai/gpt-oss-120b", model_provider="groq")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3 · batch() with max_concurrency — throttle parallel calls
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

batch_inputs = [cast(LanguageModelInput, q) for q in questions]

responses = nvidia_model.batch(
    batch_inputs,
    config={"max_concurrency": 2},   # at most 2 parallel calls at a time
)

for i, response in enumerate(responses):
    print(f"\n{'━'*60}")
    print(f"  Q{i+1}: {questions[i]}")
    print(f"{'━'*60}")
    print(f"[answer]\n{response.content}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Q1: Why do parrots have colorful feathers?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[answer]
Parrots exhibit vibrant feather colors due to a combination of evolutionary pressures tied to their social behavior, communication needs, and environmental adaptations—not a single cause. Here’s a breakdown of the key reasons, supported by ornithological research:

### 1. **Social and Sexual Selection (Primary Driver)**  
   - **Mate Attraction & Choice**: In many parrot species, bright plumage signals health, genetic quality, or foraging ability to potential mates. Unlike some birds where only males are colorful (e.g., peacocks), **both sexes of parrots are often vivid

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Q2: How do airplanes fly?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[answer]
Airplanes fly by generating enough **lift** to overcome their weight, using the princi

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangChain | Async Batch Processing — Jupyter Notebook Run
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.language_models import LanguageModelInput
from typing import cast
from dotenv import load_dotenv

load_dotenv()

# ── questions ─────────────────────────────────────────────────────────
questions  = [
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?",
]

questions = [cast(LanguageModelInput, q) for q in questions]

# ── model ─────────────────────────────────────────────────────────────
nvidia_model = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b").with_thinking_mode(enabled=True)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1 · abatch() — async version of batch(), all responses together
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

responses = await nvidia_model.abatch(questions)

for i, response in enumerate(responses):
    print(f"\n{'━'*60}")
    print(f"  Q{i+1}: {questions[i]}")
    print(f"{'━'*60}")
    print(f"[answer]\n{response.content}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Q1: Why do parrots have colorful feathers?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[answer]
Parrots exhibit exceptionally vibrant and diverse feather colors due to a combination of evolutionary pressures, primarily centered around **communication**, **species recognition**, and potentially **biochemical advantages**—rather than camouflage (which would favor duller colors in their green forest habitats). Here’s a breakdown of the key reasons:

### 1. **Sexual and Social Signaling (Mate Choice & Status)**
   - **Mate Attraction**: While many bird species show sexual dimorphism (males brighter than females), parrots often exhibit **mutual coloration**—both sexes are brightly colored. This suggests colors serve dual purposes:
     - **Courtship Displays**: Vivid plumage enhances visual signals during mating dances or rituals (e.g., the spreading of wings/tails in macaws or cockatoos).
     - **Individual R

### **Tool Calling Using LLMs**

In [1]:
# LangChain | Tool Calling 

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain.tools import tool
from dotenv import load_dotenv

load_dotenv()

nvidia_model = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b").with_thinking_mode(enabled=True)

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1 · Define tools with @tool decorator
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

@tool
def get_population(city: str) -> str:
    """Get the population of a city."""
    return f"The population of {city} is approximately 1 million."

In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2 · bind_tools() — attach tools to model
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

model_with_tools = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b").bind_tools(
    [get_weather, get_population]
)

response = model_with_tools.invoke("What's the weather like in Boston?")

print(f"\n{'━'*60}")
print(f"  Tool Calls Requested by Model")
print(f"{'━'*60}")
for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")
    print(f"ID   : {tool_call['id']}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Tool Calls Requested by Model
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Tool : get_weather
Args : {'location': 'Boston'}
ID   : chatcmpl-tool-83950ff908be8ba0


In [4]:
print(response)

content='' additional_kwargs={'reasoning_content': 'The user is asking about the weather in Boston. I have a tool called get_weather that takes a location parameter. I need to use that. The location should be "Boston". I\'ll call the get_weather function with location set to "Boston".\n', 'reasoning': 'The user is asking about the weather in Boston. I have a tool called get_weather that takes a location parameter. I need to use that. The location should be "Boston". I\'ll call the get_weather function with location set to "Boston".\n', '_reasoning_api_fields': ['reasoning_content', 'reasoning'], 'tool_calls': [{'id': 'chatcmpl-tool-83950ff908be8ba0', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}}]} response_metadata={'role': 'assistant', 'content': None, 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'chatcmpl-tool-83950ff908be8ba0', 'type': 'function', 'function': {'name': 'get_weather'

In [11]:
import sys
from pathlib import Path

project_dir = Path.cwd().resolve().parent
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from responseprint import pretty_agent_response

response_dict: dict = response if isinstance(response, dict) else {"messages": [response]}
pretty_agent_response(response_dict)


Total messages: 1

[1] AIMessage
  tool_calls:
    - get_weather({'location': 'Boston'}) id=chatcmpl-tool-83950ff908be8ba0
  tokens: in=338, out=78, total=416



In [12]:
import json

def pretty_agent_response(response):
    msgs = response["messages"] if isinstance(response, dict) else [response]
    print(f"Total messages: {len(msgs)}\n")

    for i, m in enumerate(msgs, 1):
        role = type(m).__name__
        print(f"[{i}] {role}")

        content = getattr(m, "content", "")
        if content:
            print(f"  content: {content}")

        tool_calls = getattr(m, "tool_calls", None)
        if tool_calls:
            print("  tool_calls:")
            for tc in tool_calls:
                print(f"    - {tc.get('name')}({tc.get('args')}) id={tc.get('id')}")

        tool_name = getattr(m, "name", None)
        tool_call_id = getattr(m, "tool_call_id", None)
        if tool_name or tool_call_id:
            print(f"  tool: name={tool_name}, tool_call_id={tool_call_id}")

        usage = getattr(m, "usage_metadata", None) or {}
        if usage:
            print(
                f"  tokens: in={usage.get('input_tokens')}, "
                f"out={usage.get('output_tokens')}, total={usage.get('total_tokens')}"
            )

        print()


In [13]:
pretty_agent_response({"messages": [response]})

Total messages: 1

[1] AIMessage
  tool_calls:
    - get_weather({'location': 'Boston'}) id=chatcmpl-tool-83950ff908be8ba0
  tokens: in=338, out=78, total=416



In [14]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3 · Execute tool manually & pass result back to model
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain_core.messages import BaseMessage, ToolMessage

# ─────────────────────────────────────────────────────────────────────
# STEP 1: Start with the model's first response (which contains tool calls)
# ─────────────────────────────────────────────────────────────────────
messages: list[BaseMessage] = [response]   # AIMessage with tool_calls

# The model has decided it needs to use tools.
# At this point, `response.tool_calls` is a list of dicts like:
# [
#   {"name": "get_weather", "args": {"city": "Pune"}, "id": "call_abc123"},
#   ...
# ]
# We are going to handle these calls ourselves (manual execution).

# ─────────────────────────────────────────────────────────────────────
# STEP 2: Loop through every tool call the model wants to make
# ─────────────────────────────────────────────────────────────────────
for tool_call in response.tool_calls:
    
    # 2a. Look up the actual Python function for this tool name
    #     (This is a simple dict mapping from tool name → real function)
    tool_fn = {"get_weather": get_weather, "get_population": get_population}[tool_call["name"]]
    
    # 2b. Actually run the tool with the arguments the model provided
    #     .invoke() is the LangChain way to call a tool (it handles input validation, etc.)
    result = tool_fn.invoke(tool_call["args"])
    
    # 2c. Create a ToolMessage that tells the model "here is the result of the tool"
    #     - content = whatever the tool returned (string, dict, etc.)
    #     - tool_call_id = must match the original tool call's ID so the model knows which call this result belongs to
    messages.append(
        ToolMessage(
            content=result, 
            tool_call_id=tool_call["id"]
        )
    )

# At this point, `messages` looks like:
# [
#   AIMessage( tool_calls=[ ... ] ),           ← model's first response
#   ToolMessage( content=..., tool_call_id=... ), ← result of first tool
#   ToolMessage( content=..., tool_call_id=... ), ← result of second tool (if any)
#   ...
# ]

# ─────────────────────────────────────────────────────────────────────
# STEP 3: Send everything back to the model so it can generate the final answer
# ─────────────────────────────────────────────────────────────────────
final = model_with_tools.invoke(messages)

# The model now sees:
#   - Its own previous thought (the tool calls)
#   - The actual results from the tools we just ran
#   → So it can now produce a natural final response to the user.

print(f"\n{'━'*60}")
print(f"  Final Model Response")
print(f"{'━'*60}")
print(f"[answer]\n{final.content}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Final Model Response
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[answer]
The weather in Boston is currently sunny.


In [15]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4 · Parallel tool calls — model calls multiple tools at once
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

response = model_with_tools.invoke("What's the weather and population of both SF and NYC?")

print(f"\n{'━'*60}")
print(f"  Parallel Tool Calls")
print(f"{'━'*60}")
for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}  |  Args : {tool_call['args']}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Parallel Tool Calls
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Tool : get_weather  |  Args : {'location': 'San Francisco'}


In [16]:
pretty_agent_response({"messages": [response]})

Total messages: 1

[1] AIMessage
  tool_calls:
    - get_weather({'location': 'San Francisco'}) id=chatcmpl-tool-92627f2d446d8ed6
  tokens: in=342, out=74, total=416



In [22]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SETUP — model + tools + pretty printer
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain.tools import tool
from langchain_core.messages import ToolMessage, HumanMessage, BaseMessage, SystemMessage
from dotenv import load_dotenv
import json

load_dotenv()

nvidia_model = ChatNVIDIA(
    model="nvidia/nemotron-3-super-120b-a12b"
).with_thinking_mode(enabled=True)


# ── tools ─────────────────────────────────────────────────────────────
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

@tool
def get_population(city: str) -> str:
    """Get the population of a city."""
    return f"The population of {city} is approximately 1 million."

TOOL_MAP = {
    "get_weather": get_weather,
    "get_population": get_population,
}


# ── pretty printer ────────────────────────────────────────────────────
def pretty_agent_response(response):
    msgs = response["messages"] if isinstance(response, dict) else [response]
    print(f"Total messages: {len(msgs)}\n")

    for i, m in enumerate(msgs, 1):
        role = type(m).__name__
        print(f"[{i}] {role}")

        content = getattr(m, "content", "")
        if content:
            print(f"  content: {content}")

        tool_calls = getattr(m, "tool_calls", None)
        if tool_calls:
            print("  tool_calls:")
            for tc in tool_calls:
                print(f"    - {tc.get('name')}({tc.get('args')})  id={tc.get('id')}")

        tool_name    = getattr(m, "name", None)
        tool_call_id = getattr(m, "tool_call_id", None)
        if tool_name or tool_call_id:
            print(f"  tool: name={tool_name}, tool_call_id={tool_call_id}")

        usage = getattr(m, "usage_metadata", None) or {}
        if usage:
            print(
                f"  tokens: in={usage.get('input_tokens')}, "
                f"out={usage.get('output_tokens')}, "
                f"total={usage.get('total_tokens')}"
            )
        print()

In [23]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CASE 1 — Basic bind_tools + single tool call
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"\n{'━'*60}")
print("  CASE 1 · Basic bind_tools — single tool call")
print(f"{'━'*60}")

model_with_tools = nvidia_model.bind_tools([get_weather, get_population]) # type: ignore
response = model_with_tools.invoke("What's the weather like in Boston?")

pretty_agent_response(response)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CASE 1 · Basic bind_tools — single tool call
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total messages: 1

[1] AIMessage
  tool_calls:
    - get_weather({'location': 'Boston'})  id=chatcmpl-tool-8b81a7c77c237029
  tokens: in=338, out=80, total=418



In [24]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CASE 2 — Manual tool execution loop (tool call → result → final answer)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"{'━'*60}")
print("  CASE 2 · Manual execution loop")
print(f"{'━'*60}")

messages: list[BaseMessage] = [HumanMessage(content="What's the weather in Boston?")]

# Step 1 — model decides which tool to call
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

print("── Step 1: AIMessage (tool decision) ──")
pretty_agent_response(ai_msg)

# Step 2 — execute each requested tool
for tc in ai_msg.tool_calls:
    result = TOOL_MAP[tc["name"]].invoke(tc["args"])
    messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))

print("── Step 2: ToolMessages collected ──")
for m in messages:
    if isinstance(m, ToolMessage):
        pretty_agent_response(m)

# Step 3 — model uses results to answer
final = model_with_tools.invoke(messages)

print("── Step 3: Final answer ──")
pretty_agent_response(final)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CASE 2 · Manual execution loop
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
── Step 1: AIMessage (tool decision) ──
Total messages: 1

[1] AIMessage
  tool_calls:
    - get_weather({'location': 'Boston'})  id=chatcmpl-tool-99f9930ebed05a84
  tokens: in=337, out=55, total=392

── Step 2: ToolMessages collected ──
Total messages: 1

[1] ToolMessage
  content: It's sunny in Boston.
  tool: name=None, tool_call_id=chatcmpl-tool-99f9930ebed05a84

── Step 3: Final answer ──
Total messages: 1

[1] AIMessage
  content: It's currently sunny in Boston.
  tokens: in=414, out=54, total=468



In [25]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CASE 3 — Parallel tool calls (multiple tools in one shot)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"\n{'━'*60}")
print("  CASE 3 · Parallel tool calls")
print(f"\n{'━'*60}")

response = model_with_tools.invoke(
    "What's the weather and population of both SF and NYC?"
)

print("── Parallel tool call requests ──")
pretty_agent_response(response)

# Execute all in the same loop, feed all results back at once
messages = [response]
for tc in response.tool_calls:
    result = TOOL_MAP[tc["name"]].invoke(tc["args"])
    messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))

final = model_with_tools.invoke(messages)
print("── Final answer after parallel execution ──")
pretty_agent_response(final)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CASE 3 · Parallel tool calls

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
── Parallel tool call requests ──
Total messages: 1

[1] AIMessage
  tool_calls:
    - get_weather({'location': 'San Francisco'})  id=chatcmpl-tool-86b3a48f59b1bf2b
  tokens: in=342, out=111, total=453

── Final answer after parallel execution ──
Total messages: 1

[1] AIMessage
  tool_calls:
    - get_weather({'location': 'New York City'})  id=chatcmpl-tool-a57e3fdb3bb32fcb
  tokens: in=459, out=34, total=493



In [26]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CASE 4 — Force a specific tool (tool_choice="any" / named tool)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"\n{'━'*60}")
print("  CASE 4 · Forced tool call")
print(f"\n{'━'*60}")

# Use a concrete ChatNVIDIA instance so type checkers know bind_tools() exists
base_nvidia_model = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")

# 4a — force ANY tool
model_force_any = base_nvidia_model.bind_tools(
    [get_weather, get_population],
    tool_choice="any"
)
resp_any = model_force_any.invoke("Hello!")   # model is forced to call something
print("── tool_choice='any' ──")
pretty_agent_response(resp_any)

# 4b — force a specific named tool
model_force_weather = base_nvidia_model.bind_tools(
    [get_weather, get_population],
    tool_choice="get_weather"           # only get_weather allowed
)
resp_named = model_force_weather.invoke("Tell me about Paris.")
print("── tool_choice='get_weather' (named) ──")
pretty_agent_response(resp_named)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CASE 4 · Forced tool call

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
── tool_choice='any' ──
Total messages: 1

[1] AIMessage
  tool_calls:
    - get_population({'city': 'Chicago'})  id=chatcmpl-tool-a468466027408e37
  tokens: in=332, out=51, total=383

── tool_choice='get_weather' (named) ──
Total messages: 1

[1] AIMessage
  tool_calls:
    - get_weather({'location': 'Paris'})  id=chatcmpl-tool-80c2d401913269a9
  tokens: in=335, out=148, total=483



In [29]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CASE 5 — Disable parallel tool calls
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("  CASE 5 · parallel_tool_calls=False")

base_nvidia_model = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")

model_sequential = base_nvidia_model.bind_tools(
    [get_weather, get_population],
    parallel_tool_calls=False           # at most one tool per turn
)
resp_seq = model_sequential.invoke(
    "What's the weather and population of Tokyo?"
)
print(f"Tool calls returned: {len(resp_seq.tool_calls)}  (expect ≤ 1)")
pretty_agent_response(resp_seq)

  CASE 5 · parallel_tool_calls=False
Tool calls returned: 1  (expect ≤ 1)
Total messages: 1

[1] AIMessage
  tool_calls:
    - get_weather({'location': 'Tokyo'})  id=chatcmpl-tool-a9bebccb428283f7
  tokens: in=339, out=72, total=411



In [30]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CASE 6 — Streaming tool calls (ToolCallChunk)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"{'━'*60}")
print("  CASE 6 · Streaming tool calls")
print(f"{'━'*60}")

print("── Raw chunks as they arrive ──")
for chunk in model_with_tools.stream("Weather in Boston and Tokyo?"):
    for tc in chunk.tool_call_chunks:
        if tc.get("name"):  print(f"Tool : {tc['name']}")
        if tc.get("id"):    print(f"ID   : {tc['id']}")
        if tc.get("args"):  print(f"Args : {tc['args']}")   # arrives in pieces

# ── accumulate chunks into final AIMessage ──
print("\n── Accumulated result ──")
gathered = None
for chunk in model_with_tools.stream("Weather in Boston and Tokyo?"):
    gathered = chunk if gathered is None else gathered + chunk

pretty_agent_response(gathered)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CASE 6 · Streaming tool calls
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
── Raw chunks as they arrive ──
Tool : get_weather
ID   : chatcmpl-tool-826d506756f1b64d
Args : {"location": "Boston"}

── Accumulated result ──
Total messages: 1

[1] AIMessageChunk
  tool_calls:
    - get_weather({'location': 'Boston'})  id=chatcmpl-tool-8a72baed80768aab
  tokens: in=336, out=126, total=462



In [31]:
for chunk in model_with_tools.stream(
    "What's the weather in Boston and Tokyo?"
):
    # Tool call chunks arrive progressively
    for tool_chunk in chunk.tool_call_chunks:
        if name := tool_chunk.get("name"):
            print(f"Tool: {name}")
        if id_ := tool_chunk.get("id"):
            print(f"ID: {id_}")
        if args := tool_chunk.get("args"):
            print(f"Args: {args}")

# Output:
# Tool: get_weather
# ID: call_SvMlU1TVIZugrFLckFE2ceRE
# Args: {"lo
# Args: catio
# Args: n": "B
# Args: osto
# Args: n"}
# Tool: get_weather
# ID: call_QMZdy6qInx13oWKE7KhuhOLR
# Args: {"lo
# Args: catio
# Args: n": "T
# Args: okyo
# Args: "}

Tool: get_weather
ID: chatcmpl-tool-8bb74e9ffb0e5ed5
Args: {"location": "Boston"}
